## Importing Packages

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import mlflow
import mlflow.sklearn
import mlflow.xgboost

import dagshub
import os

## DagsHub MLflow Setup

In [2]:
dagshub.init(
    repo_owner="macharumadhi",
    repo_name="DevOps-ML-Flow",
    mlflow=True
)

Initialized MLflow to track repo "macharumadhi/DevOps-ML-Flow"
Repository macharumadhi/DevOps-ML-Flow initialized!


In [3]:
mlflow.set_experiment(
    "Boston Housing Regression PBLM 1"
)

## Data Loading and Splitting

In [4]:
url = "https://raw.githubusercontent.com/selva86/datasets/master/BostonHousing.csv"
df = pd.read_csv(url)

X = df.drop(columns=["medv"])
y = df["medv"]

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42
)
print("Train size:", X_train.shape)

Train size: (354, 13)


## Build Models

In [6]:
models = [
    (
        "Linear Regression",
        LinearRegression(),
        X_train,
        y_train
    ),
    (
        "Random Forest",
        RandomForestRegressor(
            n_estimators=100,
            max_depth=5,
            random_state=42
        ),
        X_train,
        y_train
    ),
    (
        "XGBoost",
        XGBRegressor(
            n_estimators=100,
            max_depth=5,
            random_state=42
        ),
        X_train,
        y_train
    )
]

In [7]:
reports = []
trained_models = []
for model_name, model, X_tr, y_tr in models:
    model.fit(
        X_tr,
        y_tr
    )
    predictions = model.predict(
        X_test
    )
    mae = mean_absolute_error(y_test, predictions)
    mse = mean_squared_error(y_test, predictions)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, predictions)

    report = {
        "mae": mae,
        "mse": mse,
        "rmse": rmse,
        "r2": r2
    }
    reports.append(report)
    trained_models.append(model)

    print("="*50)
    print(model_name)
    print("="*50)
    print(f"R2 Score: {r2:.4f}")

Linear Regression
R2 Score: 0.7112
Random Forest
R2 Score: 0.8654
XGBoost
R2 Score: 0.8912


## Log All Experiments to DagsHub

In [8]:
for i, (model_name, model, _, _) in enumerate(models):
    report = reports[i]

    with mlflow.start_run(
        run_name=model_name
    ):
        mlflow.log_param(
            "Model",
            model_name
        )
        if hasattr(model, "get_params"):
            mlflow.log_params(
                model.get_params()
            )

        mlflow.log_metric(
            "R2_Score",
            report["r2"]
        )
        mlflow.log_metric(
            "MAE",
            report["mae"]
        )
        mlflow.log_metric(
            "MSE",
            report["mse"]
        )
        mlflow.log_metric(
            "RMSE",
            report["rmse"]
        )

        if "XGBoost" in model_name:
            mlflow.xgboost.log_model(
                model,
                "model"
            )
        else:
            mlflow.sklearn.log_model(
                model,
                "model"
            )

print("All experiments successfully logged to DagsHub!")

All experiments successfully logged to DagsHub!


## Best Model and Registry to DagsHub

In [9]:
best_index = np.argmax(
    [
        r["r2"]
        for r in reports
    ]
)
best_model_name = models[best_index][0]
best_model = trained_models[best_index]
best_report = reports[best_index]

print(
    "Best Model:",
    best_model_name
)

Best Model: XGBoost


In [10]:
with mlflow.start_run(
    run_name=f"Champion_{best_model_name}"
) as run:
    mlflow.log_param(
        "Model",
        best_model_name
    )
    mlflow.log_metric(
        "R2_Score",
        best_report["r2"]
    )
    mlflow.log_metric(
        "MAE",
        best_report["mae"]
    )
    mlflow.log_metric(
        "MSE",
        best_report["mse"]
    )
    mlflow.log_metric(
        "RMSE",
        best_report["rmse"]
    )

    if "XGBoost" in best_model_name:
        mlflow.xgboost.log_model(
            best_model,
            "model",
            registered_model_name="Boston_Housing_Best_Model"
        )
    else:
        mlflow.sklearn.log_model(
            best_model,
            "model",
            registered_model_name="Boston_Housing_Best_Model"
        )
    run_id = run.info.run_id

print("Champion Registered Run ID:", run_id)